In [7]:
!pip install -q pymorphy3 datasets conllu

In [2]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
p = morph.parse("говорила")[0]

print("lemma:", p.normal_form)
print("tag:", p.tag)

lemma: говорить
tag: VERB,impf,tran femn,sing,past,indc


In [10]:
from datasets import load_dataset

ds = load_dataset("commul/universal_dependencies", "ru_syntagrus")
print(ds)
print(ds["train"][0])

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 8906
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 8800
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 69631
    })
})
{'sent_id': '2003Anketa.xml_1', 'text': 'Анкета.', 'comments': ['__SENT_ID__', '__TEXT__'], 'tokens': ['Анкета', '.'], 'lemmas': ['анкета', '.'], 'upos': [0, 1], 'xpos': [None, None], 'feats': ['Animacy=Inan|Case=Nom|Gender=Fem|Number=Sing', None], 'head': ['0', '1'], 'deprel': ['root', 'punct'], 'deps': ['0:root', '1:punct'], 'misc': ['SpaceAfter=No', None], 'mwt': [], 'em

In [11]:
sample = ds["train"][0]
print(sample["tokens"])
print(sample["lemmas"])
print(sample["upos"])

['Анкета', '.']
['анкета', '.']
[0, 1]


In [12]:
word = sample["tokens"][0]
print("word:", word)
for parse in morph.parse(word)[:5]:
    print(parse.normal_form, parse.tag)

word: Анкета
анкета NOUN,inan,femn sing,nomn


# First ambiguity-training examples from UD + pymorphy3

In [13]:
def ud_feats_to_dict(feats):
    if feats is None:
        return {}
    if isinstance(feats, dict):
        return feats
    if isinstance(feats, str):
        out = {}
        for part in feats.split("|"):
            if "=" in part:
                k, v = part.split("=", 1)
                out[k] = v
        return out
    return {}

sample = ds["train"][0]

tokens = sample["tokens"]
lemmas = sample["lemmas"]
upos = sample["upos"]
feats = sample.get("feats", [None] * len(tokens))

examples = []

for i, word in enumerate(tokens):
    parses = morph.parse(word)

    candidate_parses = []
    for p in parses[:5]:
        candidate_parses.append({
            "lemma": p.normal_form,
            "tag": str(p.tag),
            "score": getattr(p, "score", None),
        })

    gold = {
        "word": word,
        "lemma": lemmas[i],
        "upos": upos[i],
        "feats": ud_feats_to_dict(feats[i]) if i < len(feats) else {},
    }

    # keep only potentially ambiguous words
    unique_candidates = {(c["lemma"], c["tag"]) for c in candidate_parses}
    if len(unique_candidates) > 1:
        examples.append({
            "sentence": tokens,
            "target_index": i,
            "word": word,
            "candidate_parses": candidate_parses,
            "gold": gold,
        })

print(f"Found {len(examples)} ambiguous candidates in this sentence.\n")

for ex in examples[:5]:
    print("=" * 80)
    print("WORD:", ex["word"])
    print("TARGET INDEX:", ex["target_index"])
    print("GOLD:", ex["gold"])
    print("CANDIDATES:")
    for c in ex["candidate_parses"]:
        print("  -", c)

Found 0 ambiguous candidates in this sentence.



## Run this next cell to scan more sentences until you find better ambiguous examples

In [14]:
def collect_ambiguous_examples(dataset, morph, max_sentences=100, max_examples=20):
    collected = []

    for sent_idx in range(min(max_sentences, len(dataset["train"]))):
        sample = dataset["train"][sent_idx]
        tokens = sample["tokens"]
        lemmas = sample["lemmas"]
        upos = sample["upos"]
        feats = sample.get("feats", [None] * len(tokens))

        for i, word in enumerate(tokens):
            parses = morph.parse(word)

            candidate_parses = []
            for p in parses[:5]:
                candidate_parses.append({
                    "lemma": p.normal_form,
                    "tag": str(p.tag),
                    "score": getattr(p, "score", None),
                })

            unique_candidates = {(c["lemma"], c["tag"]) for c in candidate_parses}
            if len(unique_candidates) > 1:
                collected.append({
                    "sentence": tokens,
                    "target_index": i,
                    "word": word,
                    "candidate_parses": candidate_parses,
                    "gold": {
                        "lemma": lemmas[i],
                        "upos": upos[i],
                        "feats": ud_feats_to_dict(feats[i]) if i < len(feats) else {},
                    }
                })

            if len(collected) >= max_examples:
                return collected

    return collected

ambiguous_examples = collect_ambiguous_examples(ds, morph, max_sentences=200, max_examples=10)

print(f"Collected {len(ambiguous_examples)} examples.\n")

for ex in ambiguous_examples[:3]:
    print("=" * 80)
    print("WORD:", ex["word"])
    print("SENTENCE:", " ".join(ex["sentence"]))
    print("GOLD:", ex["gold"])
    print("CANDIDATES:")
    for c in ex["candidate_parses"]:
        print("  -", c)

Collected 10 examples.

WORD: областного
SENTENCE: Начальник областного управления связи Семен Еремеевич был человек простой , приходил на работу всегда вовремя , здоровался с секретаршей за руку и иногда даже писал в стенгазету заметки под псевдонимом " Муха " .
GOLD: {'lemma': 'областной', 'upos': 6, 'feats': {'Case': 'Gen', 'Degree': 'Pos', 'Gender': 'Neut', 'Number': 'Sing'}}
CANDIDATES:
  - {'lemma': 'областной', 'tag': 'ADJF neut,sing,gent', 'score': 0.727272}
  - {'lemma': 'областной', 'tag': 'ADJF masc,sing,gent', 'score': 0.181818}
  - {'lemma': 'областной', 'tag': 'ADJF anim,masc,sing,accs', 'score': 0.090909}
WORD: управления
SENTENCE: Начальник областного управления связи Семен Еремеевич был человек простой , приходил на работу всегда вовремя , здоровался с секретаршей за руку и иногда даже писал в стенгазету заметки под псевдонимом " Муха " .
GOLD: {'lemma': 'управление', 'upos': 0, 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Neut', 'Number': 'Sing'}}
CANDIDATES

# Build a scoring function that checks whether a candidate parse matches the UD gold label.

In [17]:
upos_feature = ds["train"].features["upos"]
print(upos_feature)
print(upos_feature.feature.names)

List(ClassLabel(names=['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']))
['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']


In [18]:
upos_names = ds["train"].features["upos"].feature.names

def decode_upos(upos_value):
    if isinstance(upos_value, int):
        return upos_names[upos_value]
    return upos_value

In [19]:
for ex in ambiguous_examples:
    ex["gold"]["upos"] = decode_upos(ex["gold"]["upos"])

print(ambiguous_examples[0]["gold"])

{'lemma': 'областной', 'upos': 'ADJ', 'feats': {'Case': 'Gen', 'Degree': 'Pos', 'Gender': 'Neut', 'Number': 'Sing'}}


## Build the first scorer

In [20]:
POS_MAP = {
    "NOUN": ["NOUN"],
    "ADJ": ["ADJF", "ADJS"],
    "VERB": ["VERB", "INFN", "GRND"],
    "AUX": ["VERB", "INFN"],
    "ADV": ["ADVB", "COMP", "PRED"],
    "PRON": ["NPRO"],
    "DET": ["ADJF", "NPRO"],
    "NUM": ["NUMR"],
    "ADP": ["PREP"],
    "CCONJ": ["CONJ"],
    "SCONJ": ["CONJ"],
    "PART": ["PRCL"],
    "INTJ": ["INTJ"],
}

def parse_pymorphy_tag(tag_str):
    """
    Example:
      'NOUN,inan,neut sing,gent'
      'ADJF neut,sing,gent'
    """
    parts = tag_str.replace(",", " ").split()
    return set(parts)

def candidate_matches_gold(candidate, gold):
    cand_lemma = candidate["lemma"]
    cand_tag = candidate["tag"]
    gold_lemma = gold["lemma"]
    gold_upos = gold["upos"]

    tag_tokens = parse_pymorphy_tag(cand_tag)
    pymorphy_pos = cand_tag.split()[0].split(",")[0]

    lemma_match = (cand_lemma == gold_lemma)

    allowed_pos = POS_MAP.get(gold_upos, [gold_upos])
    pos_match = pymorphy_pos in allowed_pos

    return {
        "lemma_match": lemma_match,
        "pos_match": pos_match,
        "full_match": lemma_match and pos_match,
        "pymorphy_pos": pymorphy_pos,
        "gold_upos": gold_upos,
    }

In [21]:
ex = ambiguous_examples[0]

print("WORD:", ex["word"])
print("GOLD:", ex["gold"])
print()

for i, cand in enumerate(ex["candidate_parses"]):
    result = candidate_matches_gold(cand, ex["gold"])
    print(f"Candidate {i}: {cand}")
    print("Match result:", result)
    print()

WORD: областного
GOLD: {'lemma': 'областной', 'upos': 'ADJ', 'feats': {'Case': 'Gen', 'Degree': 'Pos', 'Gender': 'Neut', 'Number': 'Sing'}}

Candidate 0: {'lemma': 'областной', 'tag': 'ADJF neut,sing,gent', 'score': 0.727272}
Match result: {'lemma_match': True, 'pos_match': True, 'full_match': True, 'pymorphy_pos': 'ADJF', 'gold_upos': 'ADJ'}

Candidate 1: {'lemma': 'областной', 'tag': 'ADJF masc,sing,gent', 'score': 0.181818}
Match result: {'lemma_match': True, 'pos_match': True, 'full_match': True, 'pymorphy_pos': 'ADJF', 'gold_upos': 'ADJ'}

Candidate 2: {'lemma': 'областной', 'tag': 'ADJF anim,masc,sing,accs', 'score': 0.090909}
Match result: {'lemma_match': True, 'pos_match': True, 'full_match': True, 'pymorphy_pos': 'ADJF', 'gold_upos': 'ADJ'}



# Convert pymorphy3 tags into structured features and score them against UD

In [22]:
PYMORPHY_FEATURE_MAP = {
    # Case
    "nomn": ("Case", "Nom"),
    "gent": ("Case", "Gen"),
    "datv": ("Case", "Dat"),
    "accs": ("Case", "Acc"),
    "ablt": ("Case", "Ins"),
    "loct": ("Case", "Loc"),
    "loc2": ("Case", "Loc"),

    # Number
    "sing": ("Number", "Sing"),
    "plur": ("Number", "Plur"),

    # Gender
    "masc": ("Gender", "Masc"),
    "femn": ("Gender", "Fem"),
    "neut": ("Gender", "Neut"),

    # Animacy
    "anim": ("Animacy", "Anim"),
    "inan": ("Animacy", "Inan"),

    # Tense
    "past": ("Tense", "Past"),
    "pres": ("Tense", "Pres"),
    "futr": ("Tense", "Fut"),

    # Person
    "1per": ("Person", "1"),
    "2per": ("Person", "2"),
    "3per": ("Person", "3"),

    # Degree
    "COMP": ("Degree", "Cmp"),
    "Supr": ("Degree", "Sup"),
}

def pymorphy_tag_to_ud_feats(tag_str):
    feats = {}
    tokens = tag_str.replace(",", " ").split()
    for tok in tokens:
        if tok in PYMORPHY_FEATURE_MAP:
            k, v = PYMORPHY_FEATURE_MAP[tok]
            feats[k] = v
    return feats

In [23]:
for cand in ambiguous_examples[0]["candidate_parses"]:
    print(cand["tag"])
    print(pymorphy_tag_to_ud_feats(cand["tag"]))
    print()

ADJF neut,sing,gent
{'Gender': 'Neut', 'Number': 'Sing', 'Case': 'Gen'}

ADJF masc,sing,gent
{'Gender': 'Masc', 'Number': 'Sing', 'Case': 'Gen'}

ADJF anim,masc,sing,accs
{'Animacy': 'Anim', 'Gender': 'Masc', 'Number': 'Sing', 'Case': 'Acc'}



## Now build the feature scorer

In [24]:
def compare_feats(candidate_feats, gold_feats, keys=None):
    if keys is None:
        keys = sorted(set(candidate_feats.keys()) | set(gold_feats.keys()))

    result = {}
    for k in keys:
        result[k] = {
            "candidate": candidate_feats.get(k),
            "gold": gold_feats.get(k),
            "match": candidate_feats.get(k) == gold_feats.get(k)
        }
    return result

def score_candidate(candidate, gold):
    base = candidate_matches_gold(candidate, gold)

    cand_feats = pymorphy_tag_to_ud_feats(candidate["tag"])
    gold_feats = gold["feats"]

    feat_cmp = compare_feats(cand_feats, gold_feats)

    reward = 0
    if base["lemma_match"]:
        reward += 1
    if base["pos_match"]:
        reward += 1

    for feat_name, info in feat_cmp.items():
        if info["match"]:
            reward += 1

    return {
        "lemma_match": base["lemma_match"],
        "pos_match": base["pos_match"],
        "candidate_feats": cand_feats,
        "gold_feats": gold_feats,
        "feature_comparison": feat_cmp,
        "reward": reward,
    }

In [25]:
ex = ambiguous_examples[0]

print("WORD:", ex["word"])
print("GOLD:", ex["gold"])
print()

for i, cand in enumerate(ex["candidate_parses"]):
    result = score_candidate(cand, ex["gold"])
    print(f"Candidate {i}: {cand}")
    print("Reward:", result["reward"])
    print("Candidate feats:", result["candidate_feats"])
    print("Feature comparison:", result["feature_comparison"])
    print()

WORD: областного
GOLD: {'lemma': 'областной', 'upos': 'ADJ', 'feats': {'Case': 'Gen', 'Degree': 'Pos', 'Gender': 'Neut', 'Number': 'Sing'}}

Candidate 0: {'lemma': 'областной', 'tag': 'ADJF neut,sing,gent', 'score': 0.727272}
Reward: 5
Candidate feats: {'Gender': 'Neut', 'Number': 'Sing', 'Case': 'Gen'}
Feature comparison: {'Case': {'candidate': 'Gen', 'gold': 'Gen', 'match': True}, 'Degree': {'candidate': None, 'gold': 'Pos', 'match': False}, 'Gender': {'candidate': 'Neut', 'gold': 'Neut', 'match': True}, 'Number': {'candidate': 'Sing', 'gold': 'Sing', 'match': True}}

Candidate 1: {'lemma': 'областной', 'tag': 'ADJF masc,sing,gent', 'score': 0.181818}
Reward: 4
Candidate feats: {'Gender': 'Masc', 'Number': 'Sing', 'Case': 'Gen'}
Feature comparison: {'Case': {'candidate': 'Gen', 'gold': 'Gen', 'match': True}, 'Degree': {'candidate': None, 'gold': 'Pos', 'match': False}, 'Gender': {'candidate': 'Masc', 'gold': 'Neut', 'match': False}, 'Number': {'candidate': 'Sing', 'gold': 'Sing', 'ma

## find the best candidate automatically

In [26]:
def choose_best_candidate(example):
    scored = []
    for cand in example["candidate_parses"]:
        result = score_candidate(cand, example["gold"])
        scored.append({
            "candidate": cand,
            "reward": result["reward"],
            "details": result,
        })
    scored.sort(key=lambda x: x["reward"], reverse=True)
    return scored

ranked = choose_best_candidate(ambiguous_examples[0])

for item in ranked:
    print(item["reward"], item["candidate"])

5 {'lemma': 'областной', 'tag': 'ADJF neut,sing,gent', 'score': 0.727272}
4 {'lemma': 'областной', 'tag': 'ADJF masc,sing,gent', 'score': 0.181818}
3 {'lemma': 'областной', 'tag': 'ADJF anim,masc,sing,accs', 'score': 0.090909}


# Minimal OpenEnv environment skeleton

## Step 1: create a project folder

In [27]:
import os
os.makedirs("lingua_env", exist_ok=True)
os.makedirs("lingua_env/data", exist_ok=True)
os.makedirs("lingua_env/server", exist_ok=True)

## Step 2: save a tiny MVP dataset

In [59]:
!pwd
!ls
!mkdir lingua_env/data
!ls lingua_env

/home/jovyan
lingua_env  main_test.ipynb
mkdir: cannot create directory ‘lingua_env/data’: File exists
README.md    client.py	openenv.yaml		     server
__init__.py  data	openenv_lingua_env.egg-info  uv.lock
__pycache__  models.py	pyproject.toml


In [60]:
import json

mini_examples = ambiguous_examples[:50]

with open("lingua_env/data/ambiguous_examples.jsonl", "w", encoding="utf-8") as f:
    for ex in mini_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("saved", len(mini_examples), "examples")

saved 10 examples


## Step 3: define the environment shape

In [29]:
# Create a first plain Python environment class in a notebook cell before wiring it into OpenEnv:
import json
import random

class SimpleLinguaEnv:
    def __init__(self, data_path):
        with open(data_path, "r", encoding="utf-8") as f:
            self.examples = [json.loads(line) for line in f]
        self.current = None

    def reset(self):
        self.current = random.choice(self.examples)
        return {
            "sentence": self.current["sentence"],
            "target_index": self.current["target_index"],
            "word": self.current["word"],
            "candidate_parses": self.current["candidate_parses"],
        }

    def step(self, action_index):
        chosen = self.current["candidate_parses"][action_index]
        result = score_candidate(chosen, self.current["gold"])
        reward = result["reward"]
        done = True

        return {
            "sentence": self.current["sentence"],
            "target_index": self.current["target_index"],
            "word": self.current["word"],
            "candidate_parses": self.current["candidate_parses"],
        }, reward, done, {
            "chosen": chosen,
            "gold": self.current["gold"],
            "details": result,
        }

In [30]:
env = SimpleLinguaEnv("lingua_env/data/ambiguous_examples.jsonl")
obs = env.reset()

print(obs["word"])
for i, c in enumerate(obs["candidate_parses"]):
    print(i, c)

next_obs, reward, done, info = env.step(0)
print("reward:", reward)
print("done:", done)
print(info["gold"])

в
0 {'lemma': 'в', 'tag': 'PREP', 'score': 0.999327}
1 {'lemma': 'в', 'tag': 'NOUN,inan,masc,Fixd,Abbr sing,gent', 'score': 0.000249}
2 {'lemma': 'в', 'tag': 'NOUN,inan,masc,Fixd,Abbr sing,loct', 'score': 5.7e-05}
3 {'lemma': 'в', 'tag': 'NOUN,inan,masc,Fixd,Abbr sing,nomn', 'score': 1.9e-05}
4 {'lemma': 'в', 'tag': 'NOUN,inan,masc,Fixd,Abbr sing,datv', 'score': 1.9e-05}
reward: 2
done: True
{'lemma': 'в', 'upos': 'ADP', 'feats': {}}


# Wrap this into a real OpenEnv 0.2.1 environment and prepare it for HF Spaces deployment.

In [38]:
!pip install pip --upgrade
# !pip uninstall -y "openenv==0.1.13" # the required 0.2.1 not available
!pip install "openenv-core==0.2.1"

Found existing installation: openenv 0.1.13
Uninstalling openenv-0.1.13:
  Successfully uninstalled openenv-0.1.13


In [39]:
import openenv
print(openenv.__version__)

0.1.13


## 2. Freeze the MVP dataset

In [40]:
import json, os

os.makedirs("lingua_env/data", exist_ok=True)

with open("lingua_env/data/ambiguous_examples.jsonl", "w", encoding="utf-8") as f:
    for ex in ambiguous_examples[:100]:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("saved 100 examples")

saved 100 examples


## 3. Create the package skeleton

In [41]:
import os

os.makedirs("lingua_env/server", exist_ok=True)

for path in [
    "lingua_env/__init__.py",
    "lingua_env/server/__init__.py",
]:
    with open(path, "w", encoding="utf-8") as f:
        f.write("")

## 4. Write models.py

In [42]:
models_py = r'''
from pydantic import BaseModel
from typing import List, Dict, Any

class CandidateParse(BaseModel):
    lemma: str
    tag: str
    score: float | None = None

class Observation(BaseModel):
    sentence: List[str]
    target_index: int
    word: str
    candidate_parses: List[CandidateParse]

class Action(BaseModel):
    choice: int

class StepInfo(BaseModel):
    chosen: Dict[str, Any]
    gold: Dict[str, Any]
    reward_breakdown: Dict[str, Any]
'''
with open("lingua_env/server/models.py", "w", encoding="utf-8") as f:
    f.write(models_py)

print("wrote models.py")

wrote models.py


## 5. Write environment.py

In [43]:
env_py = r'''
import json
import random
from pathlib import Path

from .models import Observation

PYMORPHY_FEATURE_MAP = {
    "nomn": ("Case", "Nom"),
    "gent": ("Case", "Gen"),
    "datv": ("Case", "Dat"),
    "accs": ("Case", "Acc"),
    "ablt": ("Case", "Ins"),
    "loct": ("Case", "Loc"),
    "loc2": ("Case", "Loc"),
    "sing": ("Number", "Sing"),
    "plur": ("Number", "Plur"),
    "masc": ("Gender", "Masc"),
    "femn": ("Gender", "Fem"),
    "neut": ("Gender", "Neut"),
    "anim": ("Animacy", "Anim"),
    "inan": ("Animacy", "Inan"),
    "past": ("Tense", "Past"),
    "pres": ("Tense", "Pres"),
    "futr": ("Tense", "Fut"),
    "1per": ("Person", "1"),
    "2per": ("Person", "2"),
    "3per": ("Person", "3"),
    "COMP": ("Degree", "Cmp"),
    "Supr": ("Degree", "Sup"),
}

POS_MAP = {
    "NOUN": ["NOUN"],
    "ADJ": ["ADJF", "ADJS"],
    "VERB": ["VERB", "INFN", "GRND"],
    "AUX": ["VERB", "INFN"],
    "ADV": ["ADVB", "COMP", "PRED"],
    "PRON": ["NPRO"],
    "DET": ["ADJF", "NPRO"],
    "NUM": ["NUMR"],
    "ADP": ["PREP"],
    "CCONJ": ["CONJ"],
    "SCONJ": ["CONJ"],
    "PART": ["PRCL"],
    "INTJ": ["INTJ"],
}

def pymorphy_tag_to_ud_feats(tag_str):
    feats = {}
    tokens = tag_str.replace(",", " ").split()
    for tok in tokens:
        if tok in PYMORPHY_FEATURE_MAP:
            k, v = PYMORPHY_FEATURE_MAP[tok]
            feats[k] = v
    return feats

def parse_pymorphy_pos(tag_str):
    return tag_str.split()[0].split(",")[0]

def score_candidate(candidate, gold):
    cand_lemma = candidate["lemma"]
    cand_tag = candidate["tag"]

    gold_lemma = gold["lemma"]
    gold_upos = gold["upos"]
    gold_feats = gold.get("feats", {})

    lemma_match = cand_lemma == gold_lemma

    pymorphy_pos = parse_pymorphy_pos(cand_tag)
    allowed_pos = POS_MAP.get(gold_upos, [gold_upos])
    pos_match = pymorphy_pos in allowed_pos

    cand_feats = pymorphy_tag_to_ud_feats(cand_tag)

    feat_matches = {}
    reward = 0

    if lemma_match:
        reward += 1
    if pos_match:
        reward += 1

    for feat_name, gold_value in gold_feats.items():
        cand_value = cand_feats.get(feat_name)
        match = cand_value == gold_value
        feat_matches[feat_name] = {
            "candidate": cand_value,
            "gold": gold_value,
            "match": match,
        }
        if match:
            reward += 1

    return {
        "reward": reward,
        "lemma_match": lemma_match,
        "pos_match": pos_match,
        "candidate_feats": cand_feats,
        "gold_feats": gold_feats,
        "feature_matches": feat_matches,
    }

class LinguaEnv:
    def __init__(self, data_path=None):
        if data_path is None:
            data_path = Path(__file__).resolve().parents[1] / "data" / "ambiguous_examples.jsonl"
        self.data_path = Path(data_path)

        with open(self.data_path, "r", encoding="utf-8") as f:
            self.examples = [json.loads(line) for line in f]

        self.current = None

    def reset(self):
        self.current = random.choice(self.examples)
        return {
            "sentence": self.current["sentence"],
            "target_index": self.current["target_index"],
            "word": self.current["word"],
            "candidate_parses": self.current["candidate_parses"],
        }

    def step(self, action):
        choice = action["choice"] if isinstance(action, dict) else int(action)
        candidate = self.current["candidate_parses"][choice]
        details = score_candidate(candidate, self.current["gold"])

        obs = {
            "sentence": self.current["sentence"],
            "target_index": self.current["target_index"],
            "word": self.current["word"],
            "candidate_parses": self.current["candidate_parses"],
        }

        done = True
        reward = details["reward"]
        info = {
            "chosen": candidate,
            "gold": self.current["gold"],
            "reward_breakdown": details,
        }
        return obs, reward, done, info
'''
with open("lingua_env/server/environment.py", "w", encoding="utf-8") as f:
    f.write(env_py)

print("wrote environment.py")

wrote environment.py


## 6. Test the package before OpenEnv wiring

In [44]:
from lingua_env.server.environment import LinguaEnv

env = LinguaEnv("lingua_env/data/ambiguous_examples.jsonl")
obs = env.reset()

print("word:", obs["word"])
for i, c in enumerate(obs["candidate_parses"]):
    print(i, c)

obs2, reward, done, info = env.step(0)
print("reward:", reward)
print("done:", done)
print("gold:", info["gold"])

word: управления
0 {'lemma': 'управление', 'tag': 'NOUN,inan,neut sing,gent', 'score': 0.989071}
1 {'lemma': 'управление', 'tag': 'NOUN,inan,neut plur,nomn', 'score': 0.005464}
2 {'lemma': 'управление', 'tag': 'NOUN,inan,neut plur,accs', 'score': 0.005464}
reward: 6
done: True
gold: {'lemma': 'управление', 'upos': 'NOUN', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Neut', 'Number': 'Sing'}}


# Create the actual OpenEnv project skeleton and move your working logic into it

In [51]:
!pip install -U openenv-core uv
!rm -rf /home/jovyan/lingua_env
!openenv init lingua_env

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 146.4 MB/s  0:00:00
Creating OpenEnv environment 'lingua_env'...
✓ Created 17 files

Generating uv.lock...
✓ Generated uv.lock

Environment created successfully at: /home/jovyan/lingua_env

Next steps:
  cd /home/jovyan/lingua_env
  # Edit your environment implementation in server/lingua_env_environment.py
  # Edit your models in models.py
  # Install dependencies: uv sync

  # To integrate into OpenEnv repo:
  # 1. Copy this directory to <repo_root>/envs/lingua_env_env
  # 2. Build from repo root: docker build -t lingua_env_env:latest -f 
envs/lingua_env_env/server/Dockerfile .
  # 3. Run your image: docker run -p 8000:8000 lingua_env_env:latest


In [52]:
!find lingua_env -maxdepth 3 -type f | sort

lingua_env/README.md
lingua_env/__init__.py
lingua_env/__pycache__/__init__.cpython-313.pyc
lingua_env/__pycache__/client.cpython-313.pyc
lingua_env/__pycache__/models.cpython-313.pyc
lingua_env/client.py
lingua_env/models.py
lingua_env/openenv.yaml
lingua_env/pyproject.toml
lingua_env/server/Dockerfile
lingua_env/server/__init__.py
lingua_env/server/__pycache__/__init__.cpython-313.pyc
lingua_env/server/__pycache__/app.cpython-313.pyc
lingua_env/server/__pycache__/lingua_env_environment.cpython-313.pyc
lingua_env/server/app.py
lingua_env/server/lingua_env_environment.py
lingua_env/server/requirements.txt
lingua_env/uv.lock


## Local validation before HF Spaces

```python
# server/app.py

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
FastAPI application for the Lingua Env Environment.

This module creates an HTTP server that exposes the LinguaEnvironment
over HTTP and WebSocket endpoints, compatible with EnvClient.

Endpoints:
    - POST /reset: Reset the environment
    - POST /step: Execute an action
    - GET /state: Get current environment state
    - GET /schema: Get action/observation schemas
    - WS /ws: WebSocket endpoint for persistent sessions

Usage:
    # Development (with auto-reload):
    uvicorn server.app:app --reload --host 0.0.0.0 --port 8000

    # Production:
    uvicorn server.app:app --host 0.0.0.0 --port 8000 --workers 4

    # Or run directly:
    python -m server.app
"""

try:
    from openenv.core.env_server.http_server import create_app
except Exception as e:  # pragma: no cover
    raise ImportError(
        "openenv is required for the web interface. Install dependencies with:\n    uv sync\n"
    ) from e

from ..models import LinguaAction, LinguaObservation
from .lingua_env_environment import LinguaEnvironment

app = create_app(
    LinguaEnvironment,
    LinguaAction,
    LinguaObservation,
    env_name="lingua",
    max_concurrent_envs=1,
)


def main() -> None:
    import os
    import uvicorn

    host = os.getenv("HOST", "0.0.0.0")
    port = int(os.getenv("PORT", "8000"))
    uvicorn.run(app, host=host, port=port)


if __name__ == "__main__":
    main()

```

```bash
# File --> New --> Terminal
!uv lock
!cd lingua_env && openenv validate --verbose

# Skip the following step, 
# You do not need to run openenv build locally.
# Hugging Face Spaces builds the Docker image automatically.
!openenv build # If fails, run the following options
uv run --project . server # works
uv run server/app.py
python -m server.app

!cd ~/lingua_env
!openenv push # not notebook token, but huggingface token generated as follows:
# Step 1 — Create a new Hugging Face token
# Go to:
# https://huggingface.co/settings/tokens
# Click New token
# Use these settings:
# Name: openenv
# Role: Write
# Click Generate
# Copy the token (it will look like):
# hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

```

```bash
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Token has not been saved to git credential helper.
✓ Authenticated as: ykaitao
Preparing files for Hugging Face deployment (with web interface)...
Moved Dockerfile to repository root for deployment
✓ Updated Dockerfile: enabled web interface
Creating/verifying space: ykaitao/lingua_env
✓ Space ykaitao/lingua_env is ready
Uploading files to ykaitao/lingua_env...
✓ Upload completed successfully
Space URL: https://huggingface.co/spaces/ykaitao/lingua_env

✓ Deployment complete!
Visit your space at: https://huggingface.co/spaces/ykaitao/lingua_env

The direct endpoints should be on the Space runtime URL, typically like:

https://ykaitao-lingua-env.hf.space/health

https://ykaitao-lingua-env.hf.space/schema
```

```bash
# Test the initial env
openenv init test_env
cd test_env
openenv push
```

```bash
# File --> New --> Terminal
uv run python client_test.py
```

## From a new terminal, keep this server running and test the health/schema endpoints locally first.
```bash
curl http://127.0.0.1:8000/health &&
curl http://127.0.0.1:8000/schema &&
curl http://127.0.0.1:8000/state &&
curl -X POST http://127.0.0.1:8000/reset # real environment test

# Then choose a candidate index and test a step
curl -X POST http://127.0.0.1:8000/step \
  -H "Content-Type: application/json" \
  -d '{"action": {"choice": 0}}'
```

# GitHub to save the code

```bash
git config --global --add safe.directory /home/jovyan

git config --global user.email "ykaitao@gmail.com"
git config --global user.name "Kaitao Yang"

git remote add origin git@github.com:ykaitao/lingua_env.git
git remote -v

cd ~/lingua_env
git init
git config --global --add safe.directory /home/jovyan/lingua_env

git add .gitignore
git commit -m "add gitignore for data folder"

ssh-keygen -t ed25519 -C "ykaitao"
cat ~/.ssh/id_ed25519.pub
git remote set-url origin git@github.com:ykaitao/lingua_env.git
git push --set-upstream origin master
```